In [1]:
# Maze Solver Game for Google Colab (Playable + BFS + A*)
# Controls: buttons, plus New Maze / Reset / Solve BFS / Solve A*
# Legend: # wall, space floor, S start, G goal, @ player, · solution path

import random, math, time
from collections import deque
import heapq
import ipywidgets as widgets
from IPython.display import display

# --------------------------
# Maze generation (DFS / backtracking)
# --------------------------

DIRS = {
    "Up":    (-1, 0),
    "Down":  ( 1, 0),
    "Left":  ( 0,-1),
    "Right": ( 0, 1),
}

def carve_maze(width, height, seed=None):
    """
    Create a perfect maze on a grid of size (height x width).
    We use an odd-sized grid with walls between cells.
    """
    if seed is not None:
        random.seed(seed)

    # Ensure odd sizes for clean walls
    if width % 2 == 0: width += 1
    if height % 2 == 0: height += 1

    grid = [["#" for _ in range(width)] for _ in range(height)]

    def neighbors(cell):
        r, c = cell
        for dr, dc in [(2,0),(-2,0),(0,2),(0,-2)]:
            nr, nc = r+dr, c+dc
            if 1 <= nr < height-1 and 1 <= nc < width-1:
                yield (nr, nc, dr, dc)

    start = (1, 1)
    stack = [start]
    grid[1][1] = " "

    visited = {start}
    while stack:
        r, c = stack[-1]
        nbs = [(nr, nc, dr, dc) for (nr, nc, dr, dc) in neighbors((r,c)) if (nr, nc) not in visited]
        if not nbs:
            stack.pop()
            continue
        nr, nc, dr, dc = random.choice(nbs)
        # knock down wall between (r,c) and (nr,nc)
        grid[r + dr//2][c + dc//2] = " "
        grid[nr][nc] = " "
        visited.add((nr,nc))
        stack.append((nr,nc))

    return grid

def pick_goal(grid):
    h, w = len(grid), len(grid[0])
    # choose a reachable far-ish goal near bottom-right by scanning open cells
    open_cells = [(r,c) for r in range(1,h-1) for c in range(1,w-1) if grid[r][c] == " "]
    # prefer far from start
    open_cells.sort(key=lambda p: (p[0] + p[1]), reverse=True)
    return open_cells[0] if open_cells else (h-2, w-2)

# --------------------------
# BFS and A* solvers
# --------------------------

def reconstruct(parent, end):
    path = []
    cur = end
    while cur is not None:
        path.append(cur)
        cur = parent.get(cur)
    path.reverse()
    return path

def bfs_solve(grid, start, goal):
    h, w = len(grid), len(grid[0])
    q = deque([start])
    parent = {start: None}
    explored = 0

    while q:
        r, c = q.popleft()
        explored += 1
        if (r,c) == goal:
            path = reconstruct(parent, goal)
            return path, explored

        for dr, dc in DIRS.values():
            nr, nc = r+dr, c+dc
            if 0 <= nr < h and 0 <= nc < w and grid[nr][nc] != "#" and (nr,nc) not in parent:
                parent[(nr,nc)] = (r,c)
                q.append((nr,nc))

    return None, explored

def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar_solve(grid, start, goal):
    h, w = len(grid), len(grid[0])
    open_heap = []
    heapq.heappush(open_heap, (0 + manhattan(start, goal), 0, start))
    parent = {start: None}
    gscore = {start: 0}
    explored = 0
    in_open = {start}

    while open_heap:
        f, g, node = heapq.heappop(open_heap)
        in_open.discard(node)
        explored += 1

        if node == goal:
            path = reconstruct(parent, goal)
            return path, explored

        r, c = node
        for dr, dc in DIRS.values():
            nr, nc = r+dr, c+dc
            nxt = (nr, nc)
            if not (0 <= nr < h and 0 <= nc < w):
                continue
            if grid[nr][nc] == "#":
                continue

            tentative_g = g + 1
            if nxt not in gscore or tentative_g < gscore[nxt]:
                gscore[nxt] = tentative_g
                parent[nxt] = node
                if nxt not in in_open:
                    heapq.heappush(open_heap, (tentative_g + manhattan(nxt, goal), tentative_g, nxt))
                    in_open.add(nxt)

    return None, explored

# --------------------------
# Rendering
# --------------------------

def render(grid, start, goal, player, show_path=None):
    """
    show_path: list of (r,c) cells to draw as dots (excluding S/G).
    """
    h, w = len(grid), len(grid[0])
    path_set = set(show_path or [])

    out = []
    for r in range(h):
        row = []
        for c in range(w):
            pos = (r,c)
            if pos == player:
                row.append("@")
            elif pos == start:
                row.append("S")
            elif pos == goal:
                row.append("G")
            elif pos in path_set and grid[r][c] != "#":
                row.append("·")
            else:
                row.append(grid[r][c])
        out.append("".join(row))
    return "<pre style='font-size:15px; line-height:16px;'>" + "\n".join(out) + "</pre>"

# --------------------------
# Game state + UI
# --------------------------

class MazeGame:
    def __init__(self):
        self.grid = None
        self.start = None
        self.goal = None
        self.player = None
        self.steps = 0
        self.path_overlay = None  # last solver path (for display)
        self.last_stats = ""

    def new(self, difficulty="Medium", seed=None):
        # difficulty controls maze size
        if difficulty == "Easy":
            w, h = 21, 13
        elif difficulty == "Hard":
            w, h = 41, 21
        else:
            w, h = 31, 17

        self.grid = carve_maze(w, h, seed=seed)
        self.start = (1, 1)
        self.goal = pick_goal(self.grid)
        self.player = self.start
        self.steps = 0
        self.path_overlay = None
        self.last_stats = ""

    def reset(self):
        self.player = self.start
        self.steps = 0
        self.path_overlay = None
        self.last_stats = ""

    def move(self, dname):
        dr, dc = DIRS[dname]
        r, c = self.player
        nr, nc = r+dr, c+dc
        if self.grid[nr][nc] != "#":
            self.player = (nr,nc)
            self.steps += 1

    def solved(self):
        return self.player == self.goal

game = MazeGame()
game.new("Medium")

# Widgets
difficulty = widgets.Dropdown(
    options=["Easy", "Medium", "Hard"],
    value="Medium",
    description="Difficulty:"
)
seed_box = widgets.Text(
    value="",
    description="Seed (opt):",
    placeholder="e.g. 123"
)

board = widgets.HTML()
status = widgets.HTML()

btn_up = widgets.Button(description="↑ Up", layout=widgets.Layout(width="110px"))
btn_down = widgets.Button(description="↓ Down", layout=widgets.Layout(width="110px"))
btn_left = widgets.Button(description="← Left", layout=widgets.Layout(width="110px"))
btn_right = widgets.Button(description="→ Right", layout=widgets.Layout(width="110px"))

btn_new = widgets.Button(description="New Maze", layout=widgets.Layout(width="110px"))
btn_reset = widgets.Button(description="Reset", layout=widgets.Layout(width="110px"))
btn_bfs = widgets.Button(description="Solve BFS", layout=widgets.Layout(width="110px"))
btn_astar = widgets.Button(description="Solve A*", layout=widgets.Layout(width="110px"))
btn_clearpath = widgets.Button(description="Hide Path", layout=widgets.Layout(width="110px"))

def redraw(message=""):
    board.value = render(game.grid, game.start, game.goal, game.player, game.path_overlay)
    if game.solved():
        status.value = f"<b>✅ You reached the goal!</b> Steps: {game.steps}. {game.last_stats} {message}"
    else:
        status.value = f"Steps: {game.steps}. {game.last_stats} {message}"

def on_move(dname):
    def _(_evt):
        game.move(dname)
        redraw("")
    return _

btn_up.on_click(on_move("Up"))
btn_down.on_click(on_move("Down"))
btn_left.on_click(on_move("Left"))
btn_right.on_click(on_move("Right"))

def do_new(_evt):
    sd = seed_box.value.strip()
    seed = int(sd) if sd.isdigit() else None
    game.new(difficulty.value, seed=seed)
    redraw("<i>New maze generated.</i>")

def do_reset(_evt):
    game.reset()
    redraw("<i>Reset.</i>")

def do_bfs(_evt):
    path, explored = bfs_solve(game.grid, game.start, game.goal)
    if path is None:
        game.last_stats = "<b>⚠️ BFS:</b> No path found."
        game.path_overlay = None
    else:
        game.last_stats = f"<b>BFS:</b> path length={len(path)-1}, explored={explored}."
        game.path_overlay = [p for p in path if p not in (game.start, game.goal)]
    redraw("")

def do_astar(_evt):
    path, explored = astar_solve(game.grid, game.start, game.goal)
    if path is None:
        game.last_stats = "<b>⚠️ A*:</b> No path found."
        game.path_overlay = None
    else:
        game.last_stats = f"<b>A*:</b> path length={len(path)-1}, explored={explored}."
        game.path_overlay = [p for p in path if p not in (game.start, game.goal)]
    redraw("")

def do_clearpath(_evt):
    game.path_overlay = None
    game.last_stats = ""
    redraw("<i>Path hidden.</i>")

btn_new.on_click(do_new)
btn_reset.on_click(do_reset)
btn_bfs.on_click(do_bfs)
btn_astar.on_click(do_astar)
btn_clearpath.on_click(do_clearpath)

controls1 = widgets.HBox([btn_up])
controls2 = widgets.HBox([btn_left, btn_down, btn_right])
controls3 = widgets.HBox([btn_new, btn_reset, btn_bfs, btn_astar, btn_clearpath])

ui = widgets.VBox([
    widgets.HBox([difficulty, seed_box]),
    board,
    status,
    controls1, controls2, controls3
])

display(ui)
redraw("Reach G from S. Use buttons to move. Or click Solve BFS / Solve A*.")
